# 03 - 计划执行与优化深度教程 (Plan Execution & Refinement Deep Dive)

---
## 目录

1. [理论基础](#1-理论基础)
2. [执行上下文](#2-执行上下文)
3. [执行策略](#3-执行策略)
4. [任务执行器](#4-任务执行器)
5. [计划执行](#5-计划执行)
6. [执行监控](#6-执行监控)
7. [错误处理](#7-错误处理)
8. [计划优化](#8-计划优化)
9. [动态重规划](#9-动态重规划)
10. [实战案例](#10-实战案例)
11. [高级模式](#11-高级模式)
12. [性能优化](#12-性能优化)
13. [最佳实践](#13-最佳实践)

---

## 1. 理论基础

### 1.1 执行状态机

计划执行可以建模为有限状态机：

$$M = (Q, \Sigma, \delta, q_0, F)$$

其中：
- $Q = \{pending, running, completed, failed, blocked\}$: 状态集合
- $\Sigma = \{start, complete, fail, retry, cancel\}$: 事件集合
- $\delta: Q \times \Sigma \rightarrow Q$: 状态转移函数
- $q_0 = pending$: 初始状态
- $F = \{completed, cancelled\}$: 终止状态集合

### 1.2 执行算法

```
算法 ExecutePlan(plan, policy):
    context = InitializeContext()
    
    while not plan.is_complete:
        ready_tasks = GetReadyTasks(plan, context)
        
        if policy.parallel:
            ExecuteParallel(ready_tasks, context, policy)
        else:
            ExecuteSequential(ready_tasks, context, policy)
        
        HandleFailures(plan, context, policy)
        
        if policy.stop_on_failure and plan.has_failed_tasks:
            break
    
    return context
```

### 1.3 重规划策略

重规划优化问题：

$$\pi^* = \arg\min_{\pi \in \Pi} C(\pi) + \lambda \cdot D(\pi, \pi_{old})$$

其中：
- $\pi$: 新计划
- $\Pi$: 有效计划空间
- $C(\pi)$: 计划成本
- $D(\pi, \pi_{old})$: 与原计划的距离（稳定性）
- $\lambda$: 稳定性权重

In [ ]:
# =============================================================================
# 导入和初始化
# =============================================================================

import sys
sys.path.insert(0, '../src')

from task_decomposition import Task, TaskStatus, TaskPriority, TaskType
from plan_generation import Plan, create_planner
from plan_execution import (
    PlanExecutor, ExecutionPolicy, ExecutionContext,
    SimpleTaskExecutor, ExecutionResult, ExecutionStatus,
    ExecutionMonitor, create_executor, execute_plan
)
from plan_refinement import (
    PlanRefinement, FailureRecovery, PlanOptimizer,
    RefinementTrigger, RefinementStrategy, create_refinement
)

import json
import time
from datetime import datetime
from typing import List, Dict, Any, Callable

print("✓ 所有模块导入成功")
print(f"✓ 当前时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---

## 2. 执行上下文 (ExecutionContext)

### 2.1 ExecutionContext 数据结构

In [ ]:
# =============================================================================
# 创建执行上下文
# =============================================================================

context = ExecutionContext(
    variables={
        "project_name": "AI平台开发",
        "team_size": 10,
        "budget": 1000000,
        "deadline": "2024-12-31"
    },
    metadata={
        "start_time": datetime.now().isoformat(),
        "environment": "development"
    }
)

print("=== ExecutionContext 演示 ===")
print(f"\n变量数量: {len(context.variables)}")
print("\n共享变量:")
for key, value in context.variables.items():
    print(f"  {key}: {value}")

print(f"\n元数据: {json.dumps(context.metadata, indent=2, ensure_ascii=False)}")

In [ ]:
# =============================================================================
# 上下文操作
# =============================================================================

# 设置变量
context.set_variable("current_phase", "开发")
context.set_variable("completed_features", 5)

# 获取变量
phase = context.get_variable("current_phase")
print(f"\n当前阶段: {phase}")

# 带默认值的获取
unknown = context.get_variable("unknown_var", default="默认值")
print(f"未知变量: {unknown}")

# 添加执行结果
result1 = ExecutionResult(
    task_id="task_001",
    status=ExecutionStatus.SUCCESS,
    result="用户认证模块开发完成",
    duration=3600.0
)

result2 = ExecutionResult(
    task_id="task_002",
    status=ExecutionStatus.FAILURE,
    error="数据库连接超时",
    duration=30.0
)

context.add_result(result1)
context.add_result(result2)

print(f"\n=== 执行结果 ===")
print(f"结果总数: {len(context.results)}")

for task_id, result in context.results.items():
    status_str = "✓" if result.is_success else "✗"
    print(f"\n{status_str} {task_id}:")
    print(f"  状态: {result.status.value}")
    if result.is_success:
        print(f"  结果: {result.result}")
    else:
        print(f"  错误: {result.error}")
    print(f"  耗时: {result.duration:.2f}秒")

# 获取完成任务ID
completed_ids = context.get_completed_task_ids()
print(f"\n已完成任务: {completed_ids}")

---

## 3. 执行策略 (ExecutionPolicy)

### 3.1 策略配置

In [ ]:
# =============================================================================
# 执行策略配置
# =============================================================================

# 默认策略
default_policy = ExecutionPolicy()

print("=== 默认执行策略 ===")
print(f"\n配置:")
print(f"  max_retries: {default_policy.max_retries}")
print(f"  retry_delay: {default_policy.retry_delay}秒")
print(f"  timeout: {default_policy.timeout}秒")
print(f"  parallel: {default_policy.parallel}")
print(f"  max_workers: {default_policy.max_workers}")
print(f"  stop_on_failure: {default_policy.stop_on_failure}")

# 自定义策略
custom_policy = ExecutionPolicy(
    max_retries=5,
    retry_delay=2.0,
    timeout=600.0,
    parallel=True,
    max_workers=8,
    stop_on_failure=False  # 失败后继续
)

print("\n=== 自定义执行策略 ===")
print(f"\n配置:")
policy_dict = custom_policy.to_dict()
for key, value in policy_dict.items():
    print(f"  {key}: {value}")

### 3.2 策略对比

In [ ]:
# =============================================================================
# 不同策略对比
# =============================================================================

policies = {
    "保守策略": ExecutionPolicy(
        max_retries=1,
        stop_on_failure=True,
        parallel=False
    ),
    "标准策略": ExecutionPolicy(
        max_retries=3,
        stop_on_failure=True,
        parallel=False
    ),
    "激进策略": ExecutionPolicy(
        max_retries=5,
        stop_on_failure=False,
        parallel=True,
        max_workers=4
    ),
}

print("\n" + "="*60)
print("执行策略对比")
print("="*60)

for name, policy in policies.items():
    print(f"\n{name}:")
    print(f"  重试次数: {policy.max_retries}")
    print(f"  并行执行: {policy.parallel}")
    print(f"  失败停止: {policy.stop_on_failure}")
    print(f"  特点: ", end="")
    
    if name == "保守策略":
        print("安全优先，适合关键任务")
    elif name == "标准策略":
        print("平衡性能和可靠性")
    else:
        print("追求速度，适合容错场景")

---

## 4. 任务执行器 (TaskExecutor)

### 4.1 SimpleTaskExecutor

In [ ]:
# =============================================================================
# SimpleTaskExecutor 演示
# =============================================================================

executor = SimpleTaskExecutor()

# 注册自定义处理器
def analysis_handler(task: Task, context: ExecutionContext) -> str:
    """分析任务处理器."""
    return f"分析完成: {task.name}\n结论: 需要进一步调研"

def implementation_handler(task: Task, context: ExecutionContext) -> str:
    """实现任务处理器."""
    return f"实现完成: {task.name}\n代码行数: 1000+"

def testing_handler(task: Task, context: ExecutionContext) -> str:
    """测试任务处理器."""
    return f"测试完成: {task.name}\n覆盖率: 95%"

# 注册处理器
executor.register("analysis", analysis_handler)
executor.register("implementation", implementation_handler)
executor.register("testing", testing_handler)

# 设置默认处理器
def default_handler(task: Task, context: ExecutionContext) -> str:
    return f"默认处理: {task.name}"

executor.set_default_handler(default_handler)

# 测试执行
tasks = [
    Task(name="需求分析", task_type=TaskType.ANALYSIS),
    Task(name="API开发", task_type=TaskType.IMPLEMENTATION),
    Task(name="单元测试", task_type=TaskType.TESTING),
    Task(name="文档编写", task_type=TaskType.DOCUMENTATION),
]

context = ExecutionContext()

print("=== SimpleTaskExecutor 执行演示 ===")

for task in tasks:
    result = executor.execute(task, context)
    print(f"\n执行: {task.name} ({task.task_type.value})")
    print(f"结果:\n{result}")

### 4.2 带上下文的处理器

In [ ]:
# =============================================================================
# 上下文感知的处理器
# =============================================================================

executor = SimpleTaskExecutor()

def context_aware_handler(task: Task, context: ExecutionContext) -> str:
    """使用上下文信息的处理器."""
    # 从上下文读取变量
    project = context.get_variable("project_name", "未知项目")
    phase = context.get_variable("current_phase", "未知阶段")
    
    # 获取前置任务的结果
    if task.dependencies:
        dep_id = task.dependencies[0]
        dep_result = context.get_result(dep_id)
        dep_output = dep_result.result if dep_result else "无"
    else:
        dep_output = "无前置依赖"
    
    output = f"""项目: {project}
阶段: {phase}
任务: {task.name}
前置输出: {dep_output}

执行结果: 成功
"""
    
    # 保存结果到上下文
    context.set_variable(f"{task.id}_output", output)
    
    return output

executor.register("generic", context_aware_handler)

# 创建执行上下文
context = ExecutionContext()
context.set_variable("project_name", "智能客服系统")
context.set_variable("current_phase", "开发")

# 创建有依赖关系的任务
task1 = Task(name="数据模型设计", task_type=TaskType.ANALYSIS)
task2 = Task(name="API接口实现", task_type=TaskType.IMPLEMENTATION)
task2.add_dependency(task1.id)

print("=== 上下文感知执行演示 ===")

# 执行第一个任务
result1 = executor.execute(task1, context)
context.add_result(ExecutionResult(
    task_id=task1.id,
    status=ExecutionStatus.SUCCESS,
    result=result1
))

print("\n任务1执行结果:")
print(result1)

# 执行第二个任务（依赖第一个）
result2 = executor.execute(task2, context)

print("\n任务2执行结果:")
print(result2)

---

## 5. 计划执行 (PlanExecutor)

### 5.1 基本执行

In [ ]:
# =============================================================================
# 基本计划执行
# =============================================================================

# 创建计划
plan = create_planner("forward").generate("开发任务管理应用")

print("=== 计划执行演示 ===")
print(f"\n目标: {plan.goal}")
print(f"任务数: {len(plan.tasks)}")
print("\n执行前计划状态:")
print(format_plan(plan))

# 执行计划
executor = PlanExecutor()
context = executor.execute(plan)

print("\n执行结果:")
print(f"  计划状态: {plan.status.value}")
print(f"  完成进度: {plan.progress:.1%}")
print(f"  完成任务数: {len(context.get_completed_task_ids())}")
print(f"  总任务数: {len(plan.tasks)}")

print("\n执行后任务状态:")
for task in plan.tasks:
    status_icon = {
        TaskStatus.PENDING: "[ ]",
        TaskStatus.IN_PROGRESS: "[~]",
        TaskStatus.COMPLETED: "[x]"
    }.get(task.status, "[?]")
    print(f"  {status_icon} {task.name}: {task.status.value}")

### 5.2 顺序执行 vs 并行执行

In [ ]:
# =============================================================================
# 顺序 vs 并行执行对比
# =============================================================================

# 创建有并行可能性的计划
plan = Plan(goal="微服务开发")

# 可并行的任务
plan.add_task(Task(name="用户服务"))
plan.add_task(Task(name="订单服务"))
plan.add_task(Task(name="支付服务"))

# 有依赖的任务
plan.add_task(Task(name="API网关"))
plan.tasks[-1].add_dependency(plan.tasks[0].id)
plan.tasks[-1].add_dependency(plan.tasks[1].id)

print("\n" + "="*60)
print("执行模式对比")
print("="*60)

# 测试顺序执行
sequential_policy = ExecutionPolicy(
    parallel=False,
    max_workers=1
)

plan_copy = Plan(goal=plan.goal, tasks=plan.tasks.copy())
executor_seq = PlanExecutor(policy=sequential_policy)

start = time.time()
context_seq = executor_seq.execute(plan_copy)
sequential_time = time.time() - start

print(f"\n顺序执行:")
print(f"  执行时间: {sequential_time*1000:.2f}ms")
print(f"  完成进度: {plan_copy.progress:.1%}")

# 测试并行执行
parallel_policy = ExecutionPolicy(
    parallel=True,
    max_workers=3
)

plan_copy2 = Plan(goal=plan.goal, tasks=[Task(name=t.name) for t in plan.tasks])
executor_par = PlanExecutor(policy=parallel_policy)

start = time.time()
context_par = executor_par.execute(plan_copy2)
parallel_time = time.time() - start

print(f"\n并行执行:")
print(f"  执行时间: {parallel_time*1000:.2f}ms")
print(f"  完成进度: {plan_copy2.progress:.1%}")

speedup = sequential_time / parallel_time if parallel_time > 0 else 1
print(f"\n加速比: {speedup:.2f}x")

---

## 6. 执行监控 (ExecutionMonitor)

### 6.1 监控统计

In [ ]:
# =============================================================================
# 执行监控演示
# =============================================================================

monitor = ExecutionMonitor()
monitor.start()

# 模拟执行一些任务
tasks_executed = [
    ("task_001", 5.2, True),
    ("task_002", 3.1, True),
    ("task_003", 8.5, False),  # 失败
    ("task_004", 4.0, True),
    ("task_005", 6.3, True),
]

for task_id, duration, success in tasks_executed:
    monitor.record_task(task_id, duration, success)

# 获取统计
stats = monitor.get_stats()

print("=== 执行监控统计 ===")
print(f"\n经过时间: {stats['elapsed_time']:.2f}秒")
print(f"完成任务: {stats['tasks_completed']}")
print(f"失败任务: {stats['tasks_failed']}")
print(f"成功率: {stats['success_rate']:.1%}")
print(f"平均任务时间: {stats['average_task_time']:.2f}秒")

print("\n任务执行详情:")
for task_id, duration, success in tasks_executed:
    status = "✓" if success else "✗"
    print(f"  {status} {task_id}: {duration:.1f}秒")

### 6.2 实时监控

In [ ]:
# =============================================================================
# 实时监控演示
# =============================================================================

import time

class ProgressMonitor:
    """进度监控器."""
    
    def __init__(self, executor: PlanExecutor):
        self.executor = executor
    
    def get_progress_report(self) -> Dict[str, Any]:
        """获取进度报告."""
        stats = self.executor.monitor.get_stats()
        
        return {
            "elapsed_time": stats["elapsed_time"],
            "completed": stats["tasks_completed"],
            "failed": stats["tasks_failed"],
            "success_rate": stats["success_rate"],
            "avg_time": stats["average_task_time"],
        }

# 创建计划
plan = create_planner("hierarchical").generate("监控系统开发")

# 创建执行器和监控
executor = PlanExecutor()
progress_monitor = ProgressMonitor(executor)

print("\n=== 实时监控演示 ===")
print("\n开始执行...")

# 执行并显示进度
context = executor.execute(plan)

# 最终报告
report = progress_monitor.get_progress_report()

print("\n执行完成!")
print("\n最终报告:")
for key, value in report.items():
    if isinstance(value, float):
        if key == "success_rate":
            print(f"  {key}: {value:.1%}")
        else:
            print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

---

## 7. 错误处理

### 7.1 重试机制

In [ ]:
# =============================================================================
# 重试机制演示
# =============================================================================

class FlakyTaskExecutor(SimpleTaskExecutor):
    """模拟不稳定的任务执行器."""
    
    def __init__(self, failure_rate: float = 0.3):
        super().__init__()
        self.failure_rate = failure_rate
        self.attempt = 0
    
    def execute(self, task: Task, context: ExecutionContext) -> Any:
        import random
        self.attempt += 1
        
        # 模拟随机失败
        if random.random() < self.failure_rate:
            raise Exception(f"随机失败 (尝试 {self.attempt})")
        
        return f"任务 {task.name} 成功完成 (尝试 {self.attempt})"

# 创建不稳定的执行器
flaky_executor = FlakyTaskExecutor(failure_rate=0.5)

# 配置重试策略
retry_policy = ExecutionPolicy(
    max_retries=5,
    retry_delay=0.1,  # 快速重试
    stop_on_failure=False
)

# 创建计划
plan = Plan(goal="测试重试机制")
plan.add_task(Task(name="不稳定任务1"))
plan.add_task(Task(name="不稳定任务2"))
plan.add_task(Task(name="不稳定任务3"))

executor = PlanExecutor(
    task_executor=flaky_executor,
    policy=retry_policy
)

print("\n=== 重试机制演示 ===")
print(f"\n执行策略:")
print(f"  最大重试: {retry_policy.max_retries}")
print(f"  重试延迟: {retry_policy.retry_delay}秒")

context = executor.execute(plan)

print("\n执行结果:")
for task_id, result in context.results.items():
    status_str = "✓" if result.is_success else "✗"
    retries = result.retries
    print(f"  {status_str} {task_id}: {result.status.value} (重试{retries}次)")

# 统计
total_retries = sum(r.retries for r in context.results.values())
print(f"\n总重试次数: {total_retries}")
print(f"成功率: {context.get_completed_task_ids().__len__() / len(plan.tasks):.1%}")

### 7.2 FailureRecovery 策略

In [ ]:
# =============================================================================
# FailureRecovery 演示
# =============================================================================

recovery = FailureRecovery()

# 测试不同场景
test_cases = [
    {
        "task": Task(name="低优先级任务", priority=TaskPriority.LOW),
        "error": "资源不足",
        "attempt": 3,
        "max_attempts": 3
    },
    {
        "task": Task(name="高优先级任务", priority=TaskPriority.HIGH),
        "error": "网络超时",
        "attempt": 1,
        "max_attempts": 3
    },
    {
        "task": Task(name="关键任务", priority=TaskPriority.CRITICAL),
        "error": "服务不可用",
        "attempt": 3,
        "max_attempts": 3
    },
]

print("\n=== 失败恢复策略演示 ===")

for i, test in enumerate(test_cases, 1):
    task = test["task"]
    error = test["error"]
    attempt = test["attempt"]
    max_attempts = test["max_attempts"]
    
    strategy = recovery.suggest_recovery(
        task, error, attempt, max_attempts
    )
    
    print(f"\n测试 {i}: {task.name}")
    print(f"  优先级: {task.priority.value}")
    print(f"  尝试: {attempt}/{max_attempts}")
    print(f"  错误: {error}")
    print(f"  建议策略: {strategy.value}")
    
    # 解释策略选择
    if strategy == RefinementStrategy.RETRY:
        print(f"  理由: 还未达到最大重试次数")
    elif strategy == RefinementStrategy.SKIP:
        print(f"  理由: 低优先级任务，已达重试上限")
    elif strategy == RefinementStrategy.ABORT:
        print(f"  理由: 关键任务失败，终止执行")

---

## 8. 计划优化 (PlanOptimizer)

### 8.1 优化演示

In [ ]:
# =============================================================================
# 计划优化演示
# =============================================================================

optimizer = PlanOptimizer()

# 创建一个需要优化的计划
plan = Plan(goal="优化演示")

# 添加重复任务
plan.add_task(Task(name="设计数据库"))
plan.add_task(Task(name="设计数据库"))  # 重复

# 添加可并行的任务
plan.add_task(Task(name="前端开发"))
plan.add_task(Task(name="后端开发"))
plan.add_task(Task(name="移动端开发"))

# 添加低优先级任务
plan.add_task(Task(name="编写文档", priority=TaskPriority.LOW))

print("\n=== 计划优化演示 ===")
print(f"\n优化前:")
print(f"  任务数: {len(plan.tasks)}")
for t in plan.tasks:
    print(f"    - {t.name} ({t.priority.value})")

# 执行优化
result = optimizer.optimize(plan)

print(f"\n优化后:")
print(f"  任务数: {len(plan.tasks)}")
for t in plan.tasks:
    print(f"    - {t.name} ({t.priority.value})")

print(f"\n优化结果:")
print(f"  成功: {result.success}")
print(f"  策略: {result.strategy_used.value}")
print(f"  变更: {result.changes_made}")

# 显示并行组
if "parallel_groups" in plan.metadata:
    print(f"\n识别的并行组:")
    for i, group in enumerate(plan.metadata["parallel_groups"], 1):
        print(f"  组 {i}: {len(group)} 个任务可并行")

---

## 9. 动态重规划 (AdaptiveReplanner)

### 9.1 触发重规划

In [ ]:
# =============================================================================
# 动态重规划演示
# =============================================================================

refinement = create_refinement(max_refinements=3)

# 创建计划
plan = create_planner("forward").generate("在线教育平台")

# 模拟部分执行
plan.tasks[0].status = TaskStatus.COMPLETED
plan.tasks[1].status = TaskStatus.COMPLETED
plan.tasks[2].status = TaskStatus.FAILED  # 第3个任务失败

# 创建执行上下文
context = ExecutionContext()
context.add_result(ExecutionResult(
    task_id=plan.tasks[0].id,
    status=ExecutionStatus.SUCCESS
))
context.add_result(ExecutionResult(
    task_id=plan.tasks[1].id,
    status=ExecutionStatus.SUCCESS
))
context.add_result(ExecutionResult(
    task_id=plan.tasks[2].id,
    status=ExecutionStatus.FAILURE,
    error="API接口调用超时"
))

print("\n=== 动态重规划演示 ===")
print(f"\n当前状态:")
print(f"  计划进度: {plan.progress:.1%}")
print(f"  失败任务: {plan.tasks[2].name}")

# 处理失败
failed_task = plan.tasks[2]
recovery_result = refinement.handle_failure(
    plan=plan,
    failed_task=failed_task,
    error="API接口调用超时",
    context=context,
    attempt=1
)

print(f"\n恢复策略:")
print(f"  策略: {recovery_result.strategy_used.value}")
print(f"  成功: {recovery_result.success}")
print(f"  变更: {recovery_result.changes_made}")

---

## 10. 实战案例

### 10.1 完整工作流

In [ ]:
# =============================================================================
# 实战: CI/CD流水线执行
# =============================================================================

print("\n" + "="*60)
print("实战案例: CI/CD流水线执行与优化")
print("="*60)

# 1. 生成计划
goal = "构建完整的CI/CD流水线，包括代码检查、测试、构建、部署"

planner = create_planner("hierarchical")
plan = planner.generate(goal)

print(f"\n1. 计划生成")
print(f"   目标: {goal}")
print(f"   任务数: {len(plan.tasks)}")

# 2. 添加约束
from plan_generation import Constraint, ConstraintType

time_constraint = Constraint(
    name="部署时间窗口",
    constraint_type=ConstraintType.TIME,
    description="只能在凌晨2-4点部署",
    is_hard=True
)
plan.add_constraint(time_constraint)

print(f"\n2. 约束添加")
print(f"   约束数: {len(plan.constraints)}")

# 3. 验证计划
from plan_generation import PlanValidator

validator = PlanValidator()
validation = validator.validate(plan)

print(f"\n3. 计划验证")
print(f"   有效性: {'✓' if validation.is_valid else '✗'}")
print(f"   分数: {validation.score:.2f}")

# 4. 执行计划
print(f"\n4. 执行计划")

executor = create_executor(
    executor_type="simple",
    policy=ExecutionPolicy(
        max_retries=2,
        stop_on_failure=False
    )
)

context = executor.execute(plan)

print(f"   执行状态: {plan.status.value}")
print(f"   完成进度: {plan.progress:.1%}")

# 5. 执行监控
stats = executor.monitor.get_stats()

print(f"\n5. 执行统计")
print(f"   经过时间: {stats['elapsed_time']:.2f}秒")
print(f"   成功率: {stats['success_rate']:.1%}")
print(f"   平均耗时: {stats['average_task_time']:.2f}秒")

# 6. 如果有失败，进行优化
if plan.has_failed_tasks:
    print(f"\n6. 计划优化")
    
    refinement = create_refinement()
    
    for task in plan.tasks:
        if task.status == TaskStatus.FAILED:
            result = refinement.handle_failure(
                plan, task, task.error or "未知错误", context, 1
            )
            print(f"   {task.name}: {result.strategy_used.value}")

print("\n" + "="*60)
print("执行完成!")
print("="*60)

---

## 11. 高级模式

### 11.1 自定义回调

In [ ]:
# =============================================================================
# 自定义执行回调
# =============================================================================

class CustomCallback:
    """自定义执行回调."""
    
    def __init__(self):
        self.events = []
    
    def on_task_start(self, task: Task, context: ExecutionContext) -> None:
        self.events.append(f"[START] {task.name}")
        print(f"  → 开始: {task.name}")
    
    def on_task_complete(
        self, 
        task: Task, 
        result: ExecutionResult, 
        context: ExecutionContext
    ) -> None:
        status = "成功" if result.is_success else "失败"
        self.events.append(f"[COMPLETE] {task.name} - {status}")
        print(f"  ✓ 完成: {task.name} ({status})")
    
    def on_task_error(
        self, 
        task: Task, 
        error: Exception, 
        context: ExecutionContext
    ) -> None:
        self.events.append(f"[ERROR] {task.name} - {error}")
        print(f"  ✗ 错误: {task.name} - {error}")
    
    def on_plan_start(self, plan: Plan, context: ExecutionContext) -> None:
        self.events.append(f"[PLAN_START] {plan.goal}")
        print(f"\n▶ 开始执行计划: {plan.goal}")
    
    def on_plan_complete(self, plan: Plan, context: ExecutionContext) -> None:
        self.events.append(f"[PLAN_COMPLETE] {plan.status.value}")
        print(f"\n■ 计划完成: {plan.status.value}")

# 使用自定义回调
callback = CustomCallback()
executor = PlanExecutor(callback=callback)

plan = create_planner("forward").generate("测试回调")

print("\n=== 自定义回调演示 ===")
context = executor.execute(plan)

print(f"\n事件日志 ({len(callback.events)} 个):")
for event in callback.events:
    print(f"  {event}")

---

## 12. 性能优化

### 12.1 批量执行优化

In [ ]:
# =============================================================================
# 性能优化演示
# =============================================================================

import time

# 创建大规模计划
def create_large_plan(num_tasks: int) -> Plan:
    plan = Plan(goal=f"大规模测试 ({num_tasks} 任务)")
    for i in range(num_tasks):
        task = Task(name=f"任务_{i:04d}")
        if i > 0:
            # 创建一些依赖关系
            if i % 10 == 0:
                task.add_dependency(plan.tasks[i-1].id)
        plan.add_task(task)
    return plan

print("\n" + "="*60)
print("性能优化: 大规模计划执行")
print("="*60)

sizes = [10, 50, 100]

for size in sizes:
    plan = create_large_plan(size)
    
    # 顺序执行
    executor_seq = PlanExecutor(policy=ExecutionPolicy(parallel=False))
    start = time.time()
    context_seq = executor_seq.execute(plan)
    sequential_time = time.time() - start
    
    # 并行执行
    plan = create_large_plan(size)  # 重新创建
    executor_par = PlanExecutor(policy=ExecutionPolicy(
        parallel=True,
        max_workers=4
    ))
    start = time.time()
    context_par = executor_par.execute(plan)
    parallel_time = time.time() - start
    
    speedup = sequential_time / parallel_time if parallel_time > 0 else 1
    
    print(f"\n任务数: {size}")
    print(f"  顺序执行: {sequential_time*1000:.2f}ms")
    print(f"  并行执行: {parallel_time*1000:.2f}ms")
    print(f"  加速比: {speedup:.2f}x")

---

## 13. 最佳实践

### 13.1 执行最佳实践

In [ ]:
# =============================================================================
# 最佳实践总结
# =============================================================================

best_practices = {
    "执行策略": [
        "关键任务使用串行执行，确保可靠性",
        "独立任务使用并行执行，提高效率",
        "根据任务类型调整并行度",
    ],
    "错误处理": [
        "设置合理的重试次数 (建议 2-3 次)",
        "低优先级任务可跳过，高优先级任务必须完成",
        "关键任务失败考虑终止执行",
        "记录详细的错误信息供分析",
    ],
    "监控和日志": [
        "实时监控执行进度和状态",
        "记录关键指标 (成功率、耗时等)",
        "使用回调进行事件通知",
        "定期保存执行状态供恢复",
    ],
    "重规划策略": [
        "设置最大重规划次数避免无限循环",
        "优先修复而非重新生成整个计划",
        "保持计划稳定性，避免频繁变动",
    ],
    "性能优化": [
        "合理设置 worker 数量",
        "避免过多的任务间依赖",
        "使用批量操作减少开销",
    ],
}

print("\n" + "="*60)
print("计划执行与优化最佳实践")
print("="*60)

for category, practices in best_practices.items():
    print(f"\n{category}:")
    for i, practice in enumerate(practices, 1):
        print(f"  {i}. {practice}")

---

## 14. 总结

### 本教程覆盖的内容

1. ✓ ExecutionContext 执行上下文
2. ✓ ExecutionPolicy 执行策略
3. ✓ TaskExecutor 任务执行器
4. ✓ PlanExecutor 计划执行器
5. ✓ ExecutionMonitor 执行监控
6. ✓ 错误处理和重试机制
7. ✓ FailureRecovery 失败恢复
8. ✓ PlanOptimizer 计划优化
9. ✓ 动态重规划
10. ✓ 自定义回调
11. ✓ 性能优化技巧
12. ✓ 最佳实践

### 关键要点

- **执行上下文**: 管理共享状态和执行结果
- **执行策略**: 控制重试、并行、超时等行为
- **错误处理**: 分级处理，灵活恢复
- **监控统计**: 实时跟踪执行状态
- **动态优化**: 根据执行反馈调整计划

### 下一步学习

- 📖 **知识点.md**: 深入理解算法原理
- 📖 **测试文件**: 查看更多使用示例

---

**恭喜完成本教程! 🎉**